In [3]:
pip install requests

     ---------------------------------------- 0.0/64.7 kB ? eta -:--:--
     ---------------------------------------- 64.7/64.7 kB 3.4 MB/s eta 0:00:00
     ---------------------------------------- 0.0/154.2 kB ? eta -:--:--
     -- ------------------------------------- 10.2/154.2 kB ? eta -:--:--
     ----------------- ------------------- 71.7/154.2 kB 787.7 kB/s eta 0:00:01
     ----------------- ------------------- 71.7/154.2 kB 787.7 kB/s eta 0:00:01
     -------------------------- --------- 112.6/154.2 kB 656.4 kB/s eta 0:00:01
     --------------------------------- -- 143.4/154.2 kB 774.0 kB/s eta 0:00:01
     --------------------------------- -- 143.4/154.2 kB 774.0 kB/s eta 0:00:01
     ------------------------------------ 154.2/154.2 kB 541.4 kB/s eta 0:00:00
     ---------------------------------------- 0.0/131.6 kB ? eta -:--:--
     ------------------ -------------------- 61.4/131.6 kB 1.7 MB/s eta 0:00:01
     -------------------------------------- 131.6/131.6 kB 1.6 MB/s 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install selenium pandas webdriver-manager

     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
     -


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# ------------------ SETUP ------------------

options = Options()
options.add_argument("--start-maximized")

# Enable performance logging (important)
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

driver = webdriver.Chrome(options=options)

# ------------------ OPEN SITE ------------------

driver.get("https://www.flipkart.com")

# ------------------ GET STATUS CODE ------------------

logs = driver.get_log("performance")

status_code = None

for log in logs:
    message = log["message"]
    
    if "Network.responseReceived" in message:
        if "flipkart.com" in message:
            try:
                import json
                msg_json = json.loads(message)["message"]
                status_code = msg_json["params"]["response"]["status"]
                break
            except:
                pass

# ------------------ PRINT RESULT ------------------

if status_code:
    print("Status Code:", status_code)
else:
    print("⚠️ Could not detect status code")

driver.quit()

Status Code: 200


In [13]:
import time
import random
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------ CONFIG ------------------

SEARCH_QUERY = "iqoo"
MAX_PAGES = 3

MIN_DELAY = 3
MAX_DELAY = 6

# ------------------ SETUP ------------------

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ------------------ OPEN ------------------

driver.get("https://www.flipkart.com")
time.sleep(random.uniform(3, 5))

# ------------------ CLOSE POPUP ------------------

try:
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'✕')]"))).click()
except:
    pass

# ------------------ SEARCH ------------------

search_box = wait.until(EC.presence_of_element_located((By.NAME, "q")))
search_box.send_keys(SEARCH_QUERY)
search_box.submit()

# ✅ WAIT FOR PRODUCTS
wait.until(EC.presence_of_element_located((By.XPATH, "//div[@data-id]")))

# ------------------ SCRAPING ------------------

data = []

for page in range(1, MAX_PAGES + 1):
    print(f"\n📄 Page {page}")

    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    products = driver.find_elements(By.XPATH, "//div[@data-id]")
    print(f"📦 Found {len(products)} products")

    for item in products:

        # ---------------- NAME ----------------
        try:
            name = item.find_element(By.XPATH, ".//div[contains(@class,'RG5Slk')]").text
        except:
            name = "N/A"

        # ---------------- PRICE (FIXED) ----------------
        try:
            # Price is inside text with ₹
            price = item.find_element(By.XPATH, ".//*[contains(text(),'₹')]").text
        except:
            price = "N/A"

        # ---------------- RATING (FIXED) ----------------
        try:
            # rating is numeric like 4.3
            rating = item.find_element(By.XPATH, ".//span[contains(text(),'.')]").text
        except:
            rating = "N/A"

        # ---------------- LINK ----------------
        try:
            link = item.find_element(By.TAG_NAME, "a").get_attribute("href")
        except:
            link = "N/A"

        # ---------------- FILTER ----------------
        if name != "N/A":
            data.append({
                "Name": name,
                "Price": price,
                "Rating": rating,
                "Link": link
            })

    # ---------------- NEXT PAGE ----------------

    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(.,'Next')]")))
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(random.uniform(4, 6))
    except:
        print("❌ No more pages")
        break

# ------------------ SAVE ------------------

df = pd.DataFrame(data).drop_duplicates()
df.to_csv("flipkart_data.csv", index=False)

print("\n✅ Done")
print("📊 Total Products:", len(df))

driver.quit()


📄 Page 1
📦 Found 24 products

📄 Page 2
📦 Found 24 products

📄 Page 3
📦 Found 24 products

✅ Done
📊 Total Products: 72


In [5]:
import time
import random
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---------------- LOAD DATA ----------------

df = pd.read_csv("flipkart_data.csv")
links = df["Link"].dropna().unique()[:3]

# ---------------- SETUP ----------------

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# ---------------- SCRAPING ----------------

final_data = []

for i, link in enumerate(links):
    print(f"\n🔗 Product {i+1}")

    try:
        driver.get(link)
        time.sleep(random.uniform(5, 8))

        # scroll (IMPORTANT)
        driver.execute_script("window.scrollBy(0, 800)")
        time.sleep(2)

        page_text = driver.find_element(By.TAG_NAME, "body").text

        # -------- TITLE --------
        try:
            title = driver.find_element(By.XPATH, "//h1").text.strip()
        except:
            title = "N/A"

        # -------- PRICE --------
        price_match = re.search(r"₹\s?[\d,]+", page_text)
        price = price_match.group() if price_match else "N/A"

        # -------- RATING --------
        rating_match = re.search(r"\b\d\.\d\b", page_text)
        rating = rating_match.group() if rating_match else "N/A"

        # -------- REVIEWS --------
        reviews_match = re.search(r"[\d,]+\s+Ratings", page_text)
        reviews = reviews_match.group() if reviews_match else "N/A"

        # -------- HIGHLIGHTS --------
        highlights = []

        try:
            containers = driver.find_elements(By.XPATH, "//div[contains(@class,'_2418kt')]")

            for container in containers:
                items = container.find_elements(By.XPATH, ".//div[contains(@class,'v1zwn2k')]")

                temp = []
                for item in items:
                    text = item.text.strip()
                    if text:
                        temp.append(text)

                if 3 <= len(temp) <= 8:
                    highlights = temp
                    break

        except Exception as e:
            print("⚠️ highlight issue:", e)

        # -------- APPEND DATA (IMPORTANT) --------
        final_data.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Reviews": reviews,
            "Highlights": ", ".join(highlights),
            "Link": link
        })

        print(f"✅ Done: {title}")

    except Exception as e:
        print("❌ Error:", e)

# ---------------- SAVE ----------------

if final_data:
    final_df = pd.DataFrame(final_data)
    final_df.to_csv("flipkart_final.csv", index=False)

    print("\n🎉 SUCCESS — Data saved")
    print(final_df.head())
else:
    print("\n❌ No data collected!")

driver.quit()


🔗 Product 1
✅ Done: IQOO Z10 Lite 5G (Cyber Green, 128 GB) (6 GB RAM)

🔗 Product 2
✅ Done: IQOO Z10R 5G (Aquamarine, 128 GB) (8 GB RAM)

🔗 Product 3
✅ Done: IQOO Z10 5G (Stellar Black / Black, 128 GB) (8 GB RAM)

🎉 SUCCESS — Data saved
                                               Title    Price Rating Reviews  \
0  IQOO Z10 Lite 5G (Cyber Green, 128 GB) (6 GB RAM)  ₹14,717    4.3     N/A   
1       IQOO Z10R 5G (Aquamarine, 128 GB) (8 GB RAM)  ₹21,999    4.4     N/A   
2  IQOO Z10 5G (Stellar Black / Black, 128 GB) (8...  ₹23,079    4.4     N/A   

  Highlights                                               Link  
0             https://www.flipkart.com/iqoo-z10-lite-5g-cybe...  
1             https://www.flipkart.com/iqoo-z10r-5g-aquamari...  
2             https://www.flipkart.com/iqoo-z10-5g-stellar-b...  
